# User Modeling
## Goal 
- Building the first version of the review simulator

## importing dependencies

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [109]:
#loading the data into the notebook 
df = pd.read_csv("/kaggle/input/datasets/ravirajbabasomane/amazon-reviews-2023/Amazon_reviews_2023.csv")

In [8]:
df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True


## Data Exploration 

### Column Description

- Rating: Users provide ratings (on a scale of 1 to 5) for various products.
  
- Title:Concise titles summarizing the content of each review. 
- Text: Detailed textual descriptions of users’ experiences with the products.
- Images: Associated images (if available) related to the reviewed items. 
- ASIN (Amazon Standard Identification Number): Unique identifiers for the products.
- Parent ASIN: Identifiers for parent products (if applicable). 
- User ID: Unique identifiers for the reviewers. 
- Timestamp: Date and time when the review was posted. 
- Helpful Votes: Count of helpful votes received for each review.

- Verified Purchase: Indicates whether the reviewer made a verified purchase. Researchers and practitioners can leverage this dataset for tasks such as sentiment analysis, recommendation systems, and natural language processing. It provides valuable insights into user preferences and product quality on the Amazon platform.

In [108]:
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

=== DATASET OVERVIEW ===
Shape: (701528, 11)

Column dtypes:
rating                int64
title                object
text                 object
images               object
asin                 object
parent_asin          object
user_id              object
timestamp            object
helpful_vote          int64
verified_purchase      bool
review_seq            int64
dtype: object

Missing values:
rating                 0
title                160
text                 212
images                 0
asin                   0
parent_asin            0
user_id                0
timestamp              0
helpful_vote           0
verified_purchase      0
review_seq             0
dtype: int64

Duplicate rows: 0


### 1. Checking the Schema first 
Before anything else — lets know exactly what fields we have. Amazon Reviews schema varies by version.

In [22]:
## understanding the schema of the review dataset 
df.iloc[0].to_dict()

{'rating': 5,
 'title': 'Such a lovely scent but not overpowering.',
 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",
 'images': '[]',
 'asin': 'B00YQ6X8EO',
 'parent_asin': 'B00YQ6X8EO',
 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ',
 'timestamp': '2020-05-05 14:08:48.923',
 'helpful_vote': 0,
 'verified_purchase': True}

In [23]:
#checking the columns in the data set 
df.columns.tolist()

['rating',
 'title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase']

### 2.User history Depth 
This is our most critical check. Users with fewer than 20 reviews cannot build a believable persona

- the results here are quite worse to build the prototype i will use users with 5 or more reviews 

In [24]:
# Reviews per user

user_counts = df.groupby('user_id').size()
user_counts.describe()

count    631986.000000
mean          1.110037
std           0.753202
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         165.000000
dtype: float64

In [54]:
# How many viable user?
five_viable = user_counts[user_counts >= 5]
ten_viable = user_counts[user_counts >= 10]
twenty_viable = user_counts[user_counts >= 20]

print(f"""The number of users with five reviews or more are {len(five_viable)},
        The number of users with ten reviews or more are {len(ten_viable)},
        while the number of users with twenty reviews or more are {len(twenty_viable)}""")


The number of users with five reviews or more are 1620,
        The number of users with ten reviews or more are 330 
        while the number of users with twenty reviews or more are 117


In [29]:
# Distribution shape
user_counts.value_counts().head(20)
# expect heavy power law: that is this should be less than the above because above we are asking for greater than or equal to 5 

1     583553
2      39274
3       5713
4       1826
5        558
6        351
7        181
8        130
9         70
11        35
10        35
13        33
12        26
16        20
15        20
14        19
17        14
21         9
26         8
24         8
Name: count, dtype: int64

### 3.Rating Behaviour
we need to understand how users rate — not just globally, but per user. This feeds directly into our persona rating model.

In [34]:
# Global rating distribution
df['rating'].value_counts().sort_index()

rating
1    102080
2     43034
3     56307
4     79381
5    420726
Name: count, dtype: int64

In [36]:
#Per-user avg rating
df.groupby('user_id')['rating'].mean().describe()

count    631986.000000
mean          3.948681
std           1.487903
min           1.000000
25%           3.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: rating, dtype: float64

In [37]:
# Per-user rating variance
df.groupby('user_id')['rating'].std().describe()

count    48433.000000
mean         0.697705
std          0.918379
min          0.000000
25%          0.000000
50%          0.000000
75%          1.414214
max          2.828427
Name: rating, dtype: float64

In [40]:
#Rating over time
''' checking how users rate over time that is if a users rating is increasingly critical or generous or indifferent
     does their rating trend change? if so how helps build - persona signal

'''
# Step 1 — sort each user's reviews by time
df = df.sort_values(['user_id', 'timestamp'])

# Step 2 — assign a review sequence number per user (1st review, 2nd review, etc.)
df['review_seq'] = df.groupby('user_id').cumcount() + 1

# Step 3 — for each user, compute the slope of their rating over time
# a positive slope = they rate higher over time
# a negative slope = they become more critical over time
# near zero = consistent rater

def rating_slope(group):
    if len(group) < 5:  # need enough reviews to detect a trend
        return np.nan
    x = group['review_seq'].values
    y = group['rating'].values
    slope = np.polyfit(x, y, 1)[0]  # linear fit, take the slope
    return slope

user_slopes = df.groupby('user_id').apply(rating_slope).reset_index()
user_slopes.columns = ['user_id', 'rating_slope']

# Step 4 — look at the distribution of slopes across all users
print(user_slopes['rating_slope'].describe())

# Step 5 — classify users by their rating trend
def classify_trend(slope):
    if pd.isna(slope):
        return 'insufficient_data'
    elif slope > 0.05:
        return 'increasingly_generous'
    elif slope < -0.05:
        return 'increasingly_critical'
    else:
        return 'consistent'

user_slopes['trend'] = user_slopes['rating_slope'].apply(classify_trend)
print(user_slopes['trend'].value_counts())

count    1.620000e+03
mean    -1.484943e-02
std      2.539332e-01
min     -1.200000e+00
25%     -8.571429e-02
50%     -1.321078e-16
75%      5.714286e-02
max      1.100000e+00
Name: rating_slope, dtype: float64
trend
insufficient_data        630366
consistent                  734
increasingly_critical       475
increasingly_generous       411
Name: count, dtype: int64


In [82]:
# inspect one specific user's rating journey
sample_user = df[df['user_id'] == df['user_id'].iloc[129]]
display(sample_user[['timestamp', 'rating','review_seq', 'text']].to_string())

"                      timestamp  rating  review_seq                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             text\n242833  2016-07-0

In [101]:
# Check a 30 reviews by myself to see if whats there is really material to build a case file on a reviewer 

import textwrap

def profile_user(df, user_id=None, min_reviews=30):
    """
    Pull one user's full review history and print it as a readable story.
    If no user_id given, randomly picks a user with min_reviews+ reviews.
    """
    
    # filter to viable users
    user_counts = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index.tolist()
    
    if len(viable_users) == 0:
        print(f"No users with {min_reviews}+ reviews found.")
        return None
    
    # pick a user
    if user_id is None:
        user_id = pd.Series(viable_users).sample(1).iloc[0]
        print(f"Randomly selected user: {user_id}\n")
    elif user_id not in viable_users:
        actual_count = user_counts.get(user_id, 0)
        print(f"User {user_id} only has {actual_count} reviews. Try another.")
        return None

    # pull their reviews, sorted by time
    user_df = df[df['user_id'] == user_id].sort_values('timestamp').reset_index(drop=True)
    
    # ── HEADER ──────────────────────────────────────────────
    print("=" * 65)
    print(f"USER PROFILE: {user_id}")
    print("=" * 65)
    
    # ── QUICK STATS ─────────────────────────────────────────
    avg_rating   = user_df['rating'].mean()
    rating_std   = user_df['rating'].std()
    avg_length   = user_df['text'].str.split().str.len().mean()
    rating_dist  = user_df['rating'].value_counts().sort_index().to_dict()
    
    print(f"\n📊 QUICK STATS")
    print(f"   Total reviews   : {len(user_df)}")
    print(f"   Avg rating      : {avg_rating:.2f} ⭐  (std: {rating_std:.2f})")
    print(f"   Avg review length: {avg_length:.0f} words")
    print(f"   Rating breakdown: {rating_dist}")
    
    # ── RATING TREND ────────────────────────────────────────
    first_half = user_df.iloc[:len(user_df)//2]['rating'].mean()
    second_half = user_df.iloc[len(user_df)//2:]['rating'].mean()
    trend = "↑ more generous over time" if second_half > first_half + 0.2 \
            else "↓ more critical over time" if second_half < first_half - 0.2 \
            else "→ consistent rater"
    print(f"   Rating trend    : {trend}  (early avg: {first_half:.2f} → late avg: {second_half:.2f})")
    
    # ── CATEGORIES ──────────────────────────────────────────
    if 'category' in user_df.columns:
        top_cats = user_df['category'].value_counts().head(5).to_dict()
        print(f"\n🗂  TOP CATEGORIES")
        for cat, count in top_cats.items():
            print(f"   {cat:<35} {count} reviews")
    
    # ── FULL REVIEW HISTORY ─────────────────────────────────
    print(f"\n📝 FULL REVIEW HISTORY  (oldest → newest)")
    print("-" * 65)
    
    for i, row in user_df.iterrows():
        # convert unix timestamp to readable date
        try:
            date = pd.to_datetime(row['timestamp'], unit='s').strftime('%b %Y')
        except:
            date = "Unknown date"
        
        stars     = "⭐" * int(row['rating'])
        title   = row.get('title', '')
        review    = row.get('text', '')
        word_count = len(str(review).split())
        
        print(f"\n[{i+1}]  {date}  |  {stars} ({int(row['rating'])}/5)  |  {word_count} words")
        
        if title:
            print(f"     Headline : {title}")
        
        # wrap the review text so it's readable
        wrapped = textwrap.fill(str(review), width=60, initial_indent="     ", subsequent_indent="     ")
        print(wrapped)
        print()
    
    print("=" * 65)
    print("ASK YOURSELF:")
    print("  1. Can I describe this person in 3 sentences?")
    print("  2. Do they have a consistent complaint or praise pattern?")
    print("  3. Does their writing style feel distinctive?")
    print("  4. Would I recognise their review if I saw it without a name?")
    print("=" * 65)
    
    return user_df

In [103]:

"""
if you go through this review below you will see that you should be able to build a case file on the reviewer 
eg shes a woman, she loves body products , lives in the desert area , has a dry skin problem ,
doesnt give 5 easily, gives 4 mostly rating wise 
"""
# random user with 30+ reviews
#user_history = profile_user(df)
# or target a specific user you already found
user_history = profile_user(df, user_id='AHBWH2LBU3NFLD46GKJKIBAHKXEQ')

# or lower the bar for testing
#user_history = profile_user(df, min_reviews=20)

USER PROFILE: AHBWH2LBU3NFLD46GKJKIBAHKXEQ

📊 QUICK STATS
   Total reviews   : 39
   Avg rating      : 4.05 ⭐  (std: 1.00)
   Avg review length: 104 words
   Rating breakdown: {1: 1, 2: 3, 3: 3, 4: 18, 5: 14}
   Rating trend    : ↑ more generous over time  (early avg: 3.89 → late avg: 4.20)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Unknown date  |  ⭐⭐⭐⭐ (4/5)  |  237 words
     Headline : Good alternative to travel emery boards
     I try to avoid using traditional files and emery boards
     on my nails. I take a lot of medications that make my
     nails weak and brittle, and any surface that's too
     abrasive wreaks havoc on my fingertips. I can't carry a
     full size glass nail file with me everywhere, so I was
     on the hunt for a travel sized file that could fit in
     my pocket, wallet, or purse. When these came up, I
     thought I'd give them a shot. The pros are the files
     are a great size, the

In [104]:
# random user with 30+ reviews
user_history = profile_user(df)

Randomly selected user: AG3FVTSD7ISLKALIPY24IVJCCDTA

USER PROFILE: AG3FVTSD7ISLKALIPY24IVJCCDTA

📊 QUICK STATS
   Total reviews   : 30
   Avg rating      : 4.47 ⭐  (std: 0.63)
   Avg review length: 94 words
   Rating breakdown: {3: 2, 4: 12, 5: 16}
   Rating trend    : ↓ more critical over time  (early avg: 4.60 → late avg: 4.33)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Unknown date  |  ⭐⭐⭐⭐⭐ (5/5)  |  2 words
     Headline : Great material
     Great material.


[2]  Unknown date  |  ⭐⭐⭐⭐⭐ (5/5)  |  2 words
     Headline : Great material
     Great material.


[3]  Unknown date  |  ⭐⭐⭐ (3/5)  |  135 words
     Headline : Not really for "professional" use.
     As a professional hairstylist, I wouldn't consider this
     a "professional" straightening iron. The cord is too
     short to be used in a professional setting.<br />Also,
     it operates in Celsius and not Fahrenheit which is
     confusing to me.  It 

### 4. Review text signals
The text is where persona lives. we need to know if there's enough text per user to extract writing style.

In [56]:
# Review length distribution
df['text'].str.split().str.len().describe()

count    701316.000000
mean         32.760465
std          45.976779
min           0.000000
25%           8.000000
50%          19.000000
75%          40.000000
max        2585.000000
Name: text, dtype: float64

In [58]:
#Empty / very short reviews
# we have to filter this out cause the review is too short

df[df['text'].str.len() < 20].shape[0]

74629

In [62]:
# Avg text length per user
avg_text_per_user = df.groupby('user_id')['text'].apply(lambda x: x.str.len().mean())


In [63]:
display(avg_text_per_user)

user_id
AE222BBOVZIF42YOOPNBXL4UUMYA     52.0
AE222FP7YRNFCEQ2W3ZDIGMSYTLQ     44.0
AE222X475JC6ONXMIKZDFGQ7IAUA     14.0
AE222Y4WTST6BUZ4J5Y2H6QMBITQ    184.0
AE2232TEZOEWQLAFEX2NA6VBGMYQ     19.0
                                ...  
AHZZYVEU6QFMPFZ2HJUWR22SNK4A     11.0
AHZZZAK24AJ3JNBDUZJGHHWSRVAA    226.0
AHZZZJP24QUSB5XWW6MAXYBZZZSQ     33.0
AHZZZL7YQJA3RSA6PYK3WMFACYIQ    114.0
AHZZZSOTVOVACVK2WWXL4ITEAPIA     17.0
Name: text, Length: 631986, dtype: float64

In [64]:
#print(avg_text_per_user.value_counts())


text
9.000000       5611
12.000000      4953
13.000000      4804
11.000000      4622
10.000000      4305
               ... 
2862.000000       1
677.666667        1
108.428571        1
363.250000        1
196.600000        1
Name: count, Length: 6477, dtype: int64


In [61]:
# Verified purchase flag

df['verified_purchase'].value_counts()
# prefer verified = True

verified_purchase
True     634969
False     66559
Name: count, dtype: int64

## Combined function to get the rich-review data
this function uses the data exploration techniques above to get the rich reviews and also cleans the data 

In [110]:
import pandas as pd

def clean_amazon_reviews(df, 
                          min_reviews=20, 
                          min_word_count=30,
                          verified_only=True):
    """
    Full cleaning pipeline for Amazon Reviews dataset.
    Returns a clean dataframe of viable users ready for persona building.
    """
    
    print("=" * 60)
    print("AMAZON REVIEWS — CLEANING PIPELINE")
    print("=" * 60)
    print(f"\n▶ Starting shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # ── STEP 1: DROP DUPLICATE ROWS ─────────────────────────
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"\n[1] Duplicate rows removed   : {before - after:,}")
    print(f"    Remaining                : {after:,}")
    
    # ── STEP 2: DROP DUPLICATE COLUMNS ──────────────────────
    before_cols = df.shape[1]
    df = df.loc[:, ~df.columns.duplicated()]
    after_cols = df.shape[1]
    print(f"\n[2] Duplicate columns removed: {before_cols - after_cols}")
    print(f"    Remaining columns        : {list(df.columns)}")
    
    # ── STEP 3: REQUIRE CRITICAL COLUMNS ────────────────────
    critical_cols = ['user_id', 'rating', 'text']
    missing_cols  = [c for c in critical_cols if c not in df.columns]
    
    if missing_cols:
        print(f"\n❌ STOPPING — missing critical columns: {missing_cols}")
        print(f"   Your dataset has: {list(df.columns)}")
        return None
    
    before = len(df)
    df = df.dropna(subset=critical_cols)
    after = len(df)
    print(f"\n[3] Rows dropped (missing critical fields): {before - after:,}")
    print(f"    Remaining                              : {after:,}")
    
    # ── STEP 4: CLEAN REVIEW TEXT ───────────────────────────
    # strip whitespace
    df['text'] = df['text'].astype(str).str.strip()
    
    # remove placeholder text that slips through
    placeholders = ['n/a', 'na', 'none', 'null', '.', '-', 'no review']
    before = len(df)
    df = df[~df['text'].str.lower().isin(placeholders)]
    
    # remove very short reviews (below min_word_count)
    df['word_count'] = df['text'].str.split().str.len()
    df = df[df['word_count'] >= min_word_count]
    after = len(df)
    print(f"\n[4] Rows dropped (empty / short reviews) : {before - after:,}")
    print(f"    Min word count threshold             : {min_word_count} words")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 5: VERIFIED PURCHASES ONLY ─────────────────────
    if verified_only and 'verified_purchase' in df.columns:
        before = len(df)
        df = df[df['verified_purchase'] == True]
        after = len(df)
        print(f"\n[5] Rows dropped (unverified purchases)  : {before - after:,}")
        print(f"    Remaining                            : {after:,}")
    elif 'verified_purchase' not in df.columns:
        print(f"\n[5] Skipped — 'verified' column not found in dataset")
    
    # ── STEP 6: VALID RATINGS ONLY ──────────────────────────
    before = len(df)
    df = df[df['rating'].between(1, 5)]
    after = len(df)
    print(f"\n[6] Rows dropped (invalid ratings)       : {before - after:,}")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 7: FILTER TO USERS WITH 20+ REVIEWS ────────────
    before = len(df)
    user_counts  = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index
    df = df[df['user_id'].isin(viable_users)]
    after = len(df)
    dropped_users = len(user_counts) - len(viable_users)
    print(f"\n[7] Users dropped (< {min_reviews} reviews)          : {dropped_users:,}")
    print(f"    Viable users remaining               : {len(viable_users):,}")
    print(f"    Rows remaining                       : {after:,}")
    
    # ── STEP 8: SORT BY USER + TIME ─────────────────────────
    if 'timestamp' in df.columns:
        df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
        print(f"\n[8] Sorted by user_id + timestamp ✓")
    else:
        df = df.sort_values('user_id').reset_index(drop=True)
        print(f"\n[8] Sorted by user_id (no timestamp found) ✓")
    
    # ── FINAL SUMMARY ───────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"✅ CLEAN DATASET READY")
    print(f"{'=' * 60}")
    print(f"   Final shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Unique users     : {df['user_id'].nunique():,}")
    print(f"   Avg reviews/user : {df.groupby('user_id').size().mean():.1f}")
    print(f"   Avg word count   : {df['word_count'].mean():.0f} words")
    print(f"   Rating breakdown : {df['rating'].value_counts().sort_index().to_dict()}")
    print(f"{'=' * 60}\n")
    
    return df

In [112]:
rich_df = clean_amazon_reviews(df,min_reviews=10)

AMAZON REVIEWS — CLEANING PIPELINE

▶ Starting shape: 701,528 rows × 10 columns

[1] Duplicate rows removed   : 7,275
    Remaining                : 694,253

[2] Duplicate columns removed: 0
    Remaining columns        : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

[3] Rows dropped (missing critical fields): 211
    Remaining                              : 694,042

[4] Rows dropped (empty / short reviews) : 451,693
    Min word count threshold             : 30 words
    Remaining                            : 242,349

[5] Rows dropped (unverified purchases)  : 40,806
    Remaining                            : 201,543

[6] Rows dropped (invalid ratings)       : 0
    Remaining                            : 201,543

[7] Users dropped (< 10 reviews)          : 192,293
    Viable users remaining               : 7
    Rows remaining                       : 100

[8] Sorted by user_id + timestamp ✓

✅ CLEAN DATASET R

In [113]:
rich_df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,word_count
0,4,Doesn't leave marks on clothes!,This makeup is crazy. It comes out of the bot...,[],B07VNQ4G13,B07VNQ4G13,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-08-29 21:11:38.664,0,True,158
1,5,Colour Changing Foundation,This foundation is like no other foundation. ...,[],B07VML1QZC,B07VML1QZC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-09-27 15:08:38.780,0,True,97
2,5,Immediately see the difference...,This under eye cream is amazing. The first ti...,[],B07X1PH59J,B07X1PH59J,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-10 11:17:52.243,0,True,137
3,4,Makeup concealer,"This 3 pack of cream concealer is well, very c...",[],B07XNYVBY5,B07XNYVBY5,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-31 07:16:34.158,0,True,132
4,4,Lightweight Primer,I was surprised by how lightweight this primer...,[],B0828LCHWC,B0828LCHWC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-12-23 06:14:41.575,0,True,124


In [114]:
rich_df.to_csv('rich_users.csv', index=False)